In [1]:
%pip install -q diffusers
from mirabest.MiraBest import MiraBest
import torch
from torch.utils.data import DataLoader
from typing import Tuple
import matplotlib.pyplot as plt
import numpy as np
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import torch
import torch.nn as nn
from torch.utils.data import random_split
from ray import tune
from ray.tune import Tuner
import os
import config
from tqdm.auto import tqdm
import copy
import random
from diffusers import DDPMScheduler, UNet2DModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Setting a global seed for reproducibility
def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed(42)

In [3]:
# config
robust_training = True
vanilla_training = False
diffusion_train = False

Helper functions

In [ ]:
def get_data_loaders(dataset, transform, batch_size=2, val_split=0.2) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Returns trainloader, valloader, and testloader.
    - Splits the training data into train and validation sets.
    """
    match dataset.lower():
        case 'mirabest':
            # ---- Load full training and test sets ----
            full_train_set = MiraBest(root='./batches', train=True, download=True, transform=transform)
            testset = MiraBest(root='./batches', train=False, download=True, transform=transform)

            # ---- Create train/val split ----
            total_train_size = len(full_train_set)
            val_size = int(total_train_size * val_split)
            train_size = total_train_size - val_size

            train_subset, val_subset = random_split(full_train_set, [train_size, val_size])

            # ---- DataLoaders ----
            trainloader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=2)
            valloader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=2)
            testloader = DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)

            return trainloader, valloader, testloader
        
        case _:
            raise ValueError(f"Dataset '{dataset}' is not supported!")


def get_data(dataset,
             transform=transforms.Compose([
                 transforms.ToTensor(),  # to range [0,1]
                 transforms.Normalize([0.5], [0.5])  # 0 centers
             ])):
    """
        returns data sets
    """
    match dataset.lower():
        case 'mirabest':
            # Generate trainloader and testloader
            trainset = MiraBest(root='./batches', train=True,
                                download=True, transform=transform)
            testset = MiraBest(root='./batches', train=False,
                               download=True, transform=transform)

            return trainset, testset
        case _:
            raise ValueError(
                f'Value {dataset} does not exist in list of known datasets!')


def show_image(img):
    img = torchvision.utils.make_grid(img)
    img = img / 2 + 0.5  # denormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.show()

def pgd_attack(model, images, labels, eps=8/255, alpha=2/255, iters=10):
    device = next(model.parameters()).device

    images = images.clone().detach().to(device).float()
    labels = labels.clone().detach().to(device)
    ori_images = images.clone().detach()

    for _ in range(iters):
        images.requires_grad = True
        outputs = model(images)
        loss = F.cross_entropy(outputs, labels)
        model.zero_grad()
        loss.backward()
        grad = images.grad.sign()

        images = images + alpha * grad
        delta = torch.clamp(images - ori_images, min=-eps, max=eps)
        images = torch.clamp(ori_images + delta, min=0, max=1).detach()

    return images

def pgd_attack_early_stop(model, images, labels, eps=0.3, alpha=2/255, iters=10):
    model.eval()
    images = images.clone().detach().to(images.device)
    labels = labels.clone().detach().to(labels.device)
    ori_images = images.clone().detach()

    delta = torch.zeros_like(images, requires_grad=True)

    for i in range(iters):
        outputs = model(images + delta)
        loss = F.cross_entropy(outputs, labels)
        loss.backward()

        grad = delta.grad.detach()
        delta.data = delta + alpha * torch.sign(grad)
        delta.data = torch.clamp(delta, -eps, eps)
        delta.data = torch.clamp(ori_images + delta.data, 0, 1) - ori_images
        delta.grad.zero_()

        # Early stopping: check if attack succeeds
        with torch.no_grad():
            preds = model(images + delta).argmax(dim=1)
            if (preds != labels).all():  # All misclassified
                print(f"Fooled model on try {i}!")
                break

    adv_images = torch.clamp(images + delta.detach(), 0, 1)
    return adv_images

def optimize_parameters(config):
    print("Starting optimization")
    tuner = Tuner(
        train_,
        param_space=config,
        tune_config=tune.TuneConfig(
            metric='val_accuracy',
            mode='max',
            num_samples=30,
        ),
        run_config=tune.RunConfig(
            name=config["model_class"].__name__,
            storage_path=os.path.join(
                os.getcwd(), "tuning_results")
        ),
    )
    results = tuner.fit()
    return results


def train_(config):

    print("Training...")

    # extract inputs from config
    model_class = config["model_class"]
    model = model_class(config)

    criterion_class = config["criterion_class"]
    dataset = config["dataset"]

    criterion = criterion_class()

    # attempt to set calculation to parallel gpu training
    device, model = set_gpu_parallel_training(model)

    current_optimizer = config["optimizer_class"]
    current_lr = config["lr"]
    print(f'Using optimizer {current_optimizer}')
    print(f'Using lr {current_lr}')

    optimizer_class = current_optimizer

    # initialize optimizer function from optimizer class
    optimizer = optimizer_class(model.parameters(), lr=config["lr"])

    start_epoch = 0

    trainset, testset = get_data(dataset)

    # split dataset into training and validation sets
    test_abs = int(len(trainset) * 0.8)
    train_subset, val_subset = random_split(
        trainset,
        [test_abs, len(trainset) - test_abs]
    )

    trainloader = torch.utils.data.DataLoader(
        train_subset, batch_size=int(config['batch_size']), shuffle=True, num_workers=8
    )

    valloader = torch.utils.data.DataLoader(
        val_subset, batch_size=int(config['batch_size']), shuffle=True, num_workers=8
    )

    model.train()  # set to training mode

    training_loss = 0.0  # loss for reporting

    for epoch in range(start_epoch, 10):  # loop over the dataset multiple times
        print(f'Epoch {epoch} training...')
        running_loss = 0.0
        epoch_steps = 0
        for batch in trainloader:
            # get the inputs; data is a list of [inputs, labels]
            inputs, labels = batch
            inputs, labels = inputs.to(device), labels.to(device)

            # zero the parameter gradients
            optimizer.zero_grad()

            # forward + backward + optimize
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss += loss.item()
            epoch_steps += 1

        training_loss = running_loss / len(trainloader)

    model.eval()  # set to evaluation mode

    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in valloader:
            outputs = model(inputs)  # evaluate inputs
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        tune.report({
            "val_loss": val_loss,
            "training_loss": training_loss,
            "val_accuracy": correct/total
        })


def get_best_config(model_class, cwd):

    experimental_results_path = os.path.join(
        cwd, "tuning_results", model_class.__name__)

    print(f"Getting config from {experimental_results_path}")

    if not os.path.exists(experimental_results_path):
        raise FileExistsError(f"Parameter optimization not run for { model_class.__name__}")

    restored_tuner = tune.Tuner.restore(
        experimental_results_path, trainable=train_)

    result_grid = restored_tuner.get_results()

    return result_grid.get_best_result().config


def set_gpu_parallel_training(model):
    device = "cpu"
    if torch.cuda.is_available():
        device = "cuda:0"
        if torch.cuda.device_count() > 1:
            model = nn.DataParallel(model)

    model.to(device)

    return device, model

### Model definition

In [5]:
# define class
class ClassificationModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 34 * 34, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # conv1 output width: input_width - (kernel_size - 1) => 150 - (5-1) = 146
        # pool 1 output width: int(input_width/2) => 73
        x = self.pool(F.relu(self.conv1(x)))
        # conv2 output width: input_width - (kernel_size - 1) => 73 - (5-1) = 69
        # pool 2 output width: int(input_width/2) => 34
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 34 * 34)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

### Robust Classifier

In [6]:
classifcation_model_config = {
    'lr': tune.loguniform(1e-4, 1e-2),
    'optimizer_class': tune.choice([torch.optim.AdamW, torch.optim.Adam]),
    'model_class': ClassificationModel,
    'criterion_class': nn.CrossEntropyLoss,
    'dataset': 'MiraBest',
    'batch_size': tune.choice([8, 16])
}

# optimize if not already
current_dir = os.getcwd()
try:
    get_best_config(ClassificationModel, current_dir)
except Exception as e:
    optimize_parameters(classifcation_model_config)


Getting config from /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel


In [7]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((150, 150)),  # resize generated image
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
])

trainloader, valloader, testloader = get_data_loaders(
    "Mirabest",
    transforms.Compose([
        transforms.ToTensor(), # to range [0,1]
        transforms.Resize((150, 150)),  # resize generated image
        transforms.Normalize([0.5], [0.5]) # 0 centers
        ]),
        config.CLASSIFICATION_MODEL["batch_size"])

Files already downloaded and verified
Files already downloaded and verified


Initialize model with optimized configuration

In [8]:
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

tuned_config = get_best_config(ClassificationModel, current_dir)
adv_model = ClassificationModel(tuned_config)
criterion = tuned_config['criterion_class']()
optimizer = tuned_config['optimizer_class'](adv_model.parameters(), lr=tuned_config['lr'])
batch_size = tuned_config['batch_size']

print(tuned_config)

Getting config from /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel
{'lr': 0.0007416793445025345, 'optimizer_class': <class 'torch.optim.adam.Adam'>, 'model_class': <class '__main__.ClassificationModel'>, 'criterion_class': <class 'torch.nn.modules.loss.CrossEntropyLoss'>, 'dataset': 'MiraBest', 'batch_size': 8}


Training with early stopping

In [9]:
if robust_training:
    num_epochs = tuned_config.get("num_epochs", 10)

    T_max = 1000  # Max timestep for noise scheduling

    # === Training ===
    for epoch in range(num_epochs):
        adv_model.train()
        running_loss = 0.0

        for images, labels in tqdm(trainloader, desc=f"Epoch {epoch+1}"):
            images = images.to(device)
            labels = labels.to(device)

            # Step 1: Sample timestep t ~ Uniform[0, T]
            t = random.randint(0, T_max)

            # Step 2: Add Gaussian noise depending on timestep
            noise_std = (t / T_max) * 0.5  # Linearly scaled noise
            xt = images + torch.randn_like(images) * noise_std
            xt = torch.clamp(xt, 0, 1)

            # Step 3: Adversarial attack with early stopping on xt
            adv_images = pgd_attack_early_stop(adv_model, xt, labels, eps=8/255, alpha=2/255, iters=10)

            # Step 4: Forward pass and loss
            outputs = adv_model(adv_images)
            loss = criterion(outputs, labels)

            # Step 5: Backprop and optimize
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        avg_loss = running_loss / len(trainloader)
        print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")

    print("Finished Training")

Epoch 1:   0%|          | 0/278 [00:01<?, ?it/s]

Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled mode

Epoch 2:   0%|          | 0/278 [00:01<?, ?it/s]

Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled mode

Epoch 3:   0%|          | 0/278 [00:01<?, ?it/s]

...

Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 1 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 2 try!
Fooled model on 3 try!
Fooled model on 2 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Epoch 8 | Avg Loss: 0.4391


Epoch 9:   0%|          | 0/278 [00:01<?, ?it/s]

Fooled model on 0 try!
Fooled model on 1 try!
Fooled model on 1 try!
Fooled model on 0 try!
Fooled model on 1 try!
Fooled model on 0 try!
Fooled model on 3 try!
Fooled model on 0 try!
Fooled model on 0 try!
Epoch 9 | Avg Loss: 0.4253


Epoch 10:   0%|          | 0/278 [00:01<?, ?it/s]

Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 2 try!
Fooled model on 0 try!
Fooled model on 0 try!
Fooled model on 0 try!
Epoch 10 | Avg Loss: 0.3877
Finished Training


Evaluation

In [10]:
if robust_training:
    # get data
    dataiter = iter(testloader)
    images, labels = next(dataiter)

    # get prediction for classification model
    output = adv_model(images)
    _, predicted = torch.max(output, 1)

    # get accuracy
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            outputs = adv_model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))

Accuracy of the network on the 88 test images: 68 %


### Vanilla Classifier

In [11]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.Resize((150, 150)),  # resize generated image
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.1),
    transforms.ToTensor(),
])

trainloader, valloader, testloader = get_data_loaders(
    "Mirabest",
    transforms.Compose([
        transforms.ToTensor(), # to range [0,1]
        transforms.Resize((150, 150)),  # resize generated image
        transforms.Normalize([0.5], [0.5]) # 0 centers
        ]),
        config.CLASSIFICATION_MODEL["batch_size"])

Files already downloaded and verified
Files already downloaded and verified


In [12]:
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

tuned_config = get_best_config(ClassificationModel, current_dir)
van_model = ClassificationModel(tuned_config)
criterion = tuned_config['criterion_class']()
optimizer = tuned_config['optimizer_class'](adv_model.parameters(), lr=tuned_config['lr'])
batch_size = tuned_config['batch_size']

print(tuned_config)

Getting config from /Users/bevanslabbert/Documents/GitHub/pid-radast/tuning_results/ClassificationModel
{'lr': 0.0007416793445025345, 'optimizer_class': <class 'torch.optim.adam.Adam'>, 'model_class': <class '__main__.ClassificationModel'>, 'criterion_class': <class 'torch.nn.modules.loss.CrossEntropyLoss'>, 'dataset': 'MiraBest', 'batch_size': 8}


In [13]:
if vanilla_training:
    current_dir = os.getcwd()

    tuned_config = get_best_config(ClassificationModel, current_dir)
    van_model = ClassificationModel(tuned_config)
    criterion = tuned_config['criterion_class']()
    optimizer = tuned_config['optimizer_class'](van_model.parameters(), lr=tuned_config['lr'])
    batch_size = tuned_config['batch_size']

    for epoch in range(config.CLASSIFICATION_MODEL["num_epochs"]):

        running_loss = 0.0
        for data in tqdm(trainloader, 0):
            # get the inputs
            images, labels = data

            # forward on clean and adversarial
            outputs_clean = van_model(images)

            # combined loss
            loss_clean = F.cross_entropy(outputs_clean, labels)
            loss = loss_clean

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"epoch {epoch} | loss: {running_loss/len(trainloader):.4f}")

    print('finished training')


Evaluation

In [14]:
if vanilla_training:
    # get data
    dataiter = iter(testloader)
    images, labels = next(dataiter)

    # get prediction for classification model
    output = van_model(images)
    _, predicted = torch.max(output, 1)

    # get accuracy
    correct = 0
    total = 0
    with torch.no_grad():
        for data in testloader:
            images, labels = data
            outputs = van_model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print('Accuracy of the network on the 88 test images: %d %%' % (100 * correct / total))

### Diffusion Model

In [15]:
class ClassConditionedUnet(nn.Module):
    def __init__(self, num_classes=10, class_emb_size=4):
        super().__init__()

        # The embedding layer will map the class label to a vector of size class_emb_size
        self.class_emb = nn.Embedding(num_classes, class_emb_size)

        # Self.model is an unconditional UNet with extra input channels to accept the conditioning information (the class embedding)
        self.model = UNet2DModel(
            sample_size=150,  # the target image resolution
            in_channels=1 + class_emb_size,  # Additional input channels for class cond.
            out_channels=1,  # the number of output channels
            layers_per_block=2,  # how many ResNet layers to use per UNet block
            block_out_channels=(32, 64, 64),
            down_block_types=(
                "DownBlock2D",  # a regular ResNet downsampling block
                "AttnDownBlock2D",  # a ResNet downsampling block with spatial self-attention
                "AttnDownBlock2D",
            ),
            up_block_types=(
                "AttnUpBlock2D",
                "AttnUpBlock2D",  # a ResNet upsampling block with spatial self-attention
                "UpBlock2D",  # a regular ResNet upsampling block
            ),
        )

    # Our forward method now takes the class labels as an additional argument
    def forward(self, x, t, class_labels):
        # Shape of x:
        bs, ch, w, h = x.shape

        # class conditioning in right shape to add as additional input channels
        class_cond = self.class_emb(class_labels)  # Map to embedding dimension
        class_cond = class_cond.view(bs, class_cond.shape[1], 1, 1).expand(bs, class_cond.shape[1], w, h)
        # x is shape (bs, 1, 28, 28) and class_cond is now (bs, 4, 28, 28)

        # Net input is now x and class cond concatenated together along dimension 1
        net_input = torch.cat((x, class_cond), 1)  # (bs, 5, 28, 28)

        # Feed this to the UNet alongside the timestep and return the prediction
        return self.model(net_input, t).sample  # (bs, 1, 28, 28)

In [16]:
transform = transforms.Compose([
    transforms.Resize((152, 152)),
    transforms.ToTensor()
])

trainloader, valloader, testloader = get_data_loaders("mirabest", transform)

Files already downloaded and verified
Files already downloaded and verified


In [17]:
if diffusion_train:
    model = ClassConditionedUnet().to(device)
    noise_scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")

    loss_fn = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    losses = []

    # The training loop
    for epoch in range(config.DIFFUSION_MODEL["num_epochs"]):
        for x, y in tqdm(trainloader):

            # Get some data and prepare the corrupted version
            x = x.to(device) * 2 - 1  # Data on the GPU (mapped to (-1, 1))
            y = y.to(device)
            noise = torch.randn_like(x)
            timesteps = torch.randint(0, 999, (x.shape[0],)).long().to(device)
            noisy_x = noise_scheduler.add_noise(x, noise, timesteps)

            # Get the model prediction
            pred = model(noisy_x, timesteps, y)  # Note that we pass in the labels y

            # Calculate the loss
            loss = loss_fn(pred, noise)  # How close is the output to the noise

            # Backprop and update the params:
            opt.zero_grad()
            loss.backward()
            opt.step()

            # Store the loss for later
            losses.append(loss.item())

        # Print out the average of the last 100 loss values to get an idea of progress:
        avg_loss = sum(losses[-100:]) / 100
        print(f"Finished epoch {epoch}. Average of the last 100 loss values: {avg_loss:05f}")


### Integration